## Skin Cancer Classification HAM10000 - EfficientNetV2S Frozen Backbone vs Fine-Tuned

Model A: EfficientNetV2S pretrained ImageNet, backbone dibekukan, hanya head yang dilatih (tanpa augmentasi dan class weight).

Model B: EfficientNetV2S two-phase transfer learning (fine-tuning) + augmentasi + class weight, learning rate fase 2 dipilih dengan 5-fold CV.

Data dibagi per lesi (`lesion_id`) supaya citra dari lesi yang sama tidak masuk ke train dan test sekaligus.

### Import Library

In [ ]:
import os
import gc
import json
import time
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix,
                             classification_report, roc_auc_score)
from scipy.stats import binomtest, chi2

### Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder project di Google Drive
ROOT_DIR = Path('/content/drive/MyDrive/skin-cancer')
HAM_DIR = ROOT_DIR / 'data' / 'HAM10000'
METADATA_CSV = HAM_DIR / 'HAM10000_metadata.csv'
IMAGE_DIRS = [HAM_DIR / 'HAM10000_images_part_1', HAM_DIR / 'HAM10000_images_part_2']

SPLIT_DIR = ROOT_DIR / 'data' / 'splits'
MODEL_DIR = ROOT_DIR / 'models'
METRICS_DIR = ROOT_DIR / 'results' / 'metrics'
HISTORY_DIR = ROOT_DIR / 'results' / 'history'
CHECKPOINT_DIR = ROOT_DIR / 'checkpoints'
OUT_DIR = ROOT_DIR / 'konferensi'
A_DIR = OUT_DIR / 'A_effv2s_tanpa_finetuning'
FIG_DIR = OUT_DIR / 'figures'

for d in [SPLIT_DIR, MODEL_DIR, METRICS_DIR, HISTORY_DIR, CHECKPOINT_DIR,
          A_DIR / 'cv', A_DIR / 'final', FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed()

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)
print(tf.config.list_physical_devices('GPU'))

### Load Dataset

In [ ]:
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
NUM_CLASSES = len(CLASS_NAMES)
LABELS = list(range(NUM_CLASSES))
MEL = CLASS_NAMES.index('mel')

df = pd.read_csv(METADATA_CSV)

image_paths = {}
for d in IMAGE_DIRS:
    for f in os.listdir(d):
        if f.endswith('.jpg'):
            image_paths[f[:-4]] = str(d / f)

df['path'] = df['image_id'].map(image_paths)
df['label'] = df['dx'].map({c: i for i, c in enumerate(CLASS_NAMES)})
print(len(df), 'citra,', df['lesion_id'].nunique(), 'lesi')
df.head()

In [ ]:
counts = df['dx'].value_counts().reindex(CLASS_NAMES)
pd.DataFrame({'jumlah': counts, 'persen': (counts / len(df) * 100).round(2)})

### Train Test Split (per lesi)

In [ ]:
train_csv = SPLIT_DIR / 'train.csv'
test_csv = SPLIT_DIR / 'test.csv'

if train_csv.exists() and test_csv.exists():
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
else:
    # 80:20, dikelompokkan berdasarkan lesion_id
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, test_idx = next(sgkf.split(df, df['label'], groups=df['lesion_id']))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)

print('Train:', len(train_df), '| Test:', len(test_df))
print('Lesi yang ada di train dan test:', len(set(train_df['lesion_id']) & set(test_df['lesion_id'])))

In [ ]:
N_FOLDS = 5
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(train_df, train_df['label'].values, groups=train_df['lesion_id']))

pd.DataFrame([{'fold': i, 'train': len(tr), 'val': len(va),
               **train_df.iloc[va]['dx'].value_counts().reindex(CLASS_NAMES).to_dict()}
              for i, (tr, va) in enumerate(folds, 1)])

### Data Pipeline

In [ ]:
IMG_SIZE = (384, 384)
BATCH_SIZE = 16
preprocess = keras.applications.efficientnet_v2.preprocess_input


def load_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = tf.cast(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img


class BrightnessMultiply(keras.layers.Layer):
    # brightness dikali faktor acak 0.8 - 1.2 (seperti brightness_range di ImageDataGenerator)
    def __init__(self, low=0.8, high=1.2, seed=None, **kwargs):
        super().__init__(**kwargs)
        self.low = low
        self.high = high
        self.seed = seed

    def call(self, x, training=False):
        if not training:
            return x
        factor = tf.random.uniform((tf.shape(x)[0], 1, 1, 1), self.low, self.high,
                                   dtype=x.dtype, seed=self.seed)
        return tf.clip_by_value(x * factor, 0.0, 255.0)

    def compute_output_shape(self, input_shape):
        return input_shape


def make_augmentation():
    return keras.Sequential([
        keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        keras.layers.RandomRotation(20 / 360, fill_mode='nearest', seed=SEED),
        keras.layers.RandomZoom(0.2, 0.2, fill_mode='nearest', seed=SEED),
        keras.layers.RandomTranslation(0.1, 0.1, fill_mode='nearest', seed=SEED),
        BrightnessMultiply(0.8, 1.2, seed=SEED),
    ])


def make_dataset(data, augment=False, shuffle=False):
    labels = keras.utils.to_categorical(data['label'].values, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((data['path'].astype(str).values, labels))
    if shuffle:
        ds = ds.shuffle(min(len(data), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        aug = make_augmentation()
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def get_class_weight(data):
    # w_c = N / (K * n_c)
    n = data['dx'].value_counts().reindex(CLASS_NAMES)
    return {i: len(data) / (NUM_CLASSES * n[c]) for i, c in enumerate(CLASS_NAMES)}


pd.Series(get_class_weight(train_df)).rename(dict(enumerate(CLASS_NAMES))).round(3)

### Model

In [ ]:
def build_model():
    base = keras.applications.EfficientNetV2S(include_top=False, weights='imagenet',
                                              input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(base.output)
    x = keras.layers.Dense(256, activation='relu', name='dense_256')(x)
    x = keras.layers.Dropout(0.5, name='dropout')(x)
    out = keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='predictions')(x)
    return keras.Model(base.input, out, name='EfficientNetV2S_TL')


def set_trainable(model, train_backbone):
    # 4 layer terakhir adalah head, BatchNorm tetap beku saat fine-tuning
    for layer in model.layers[:-4]:
        if train_backbone and isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = train_backbone
    for layer in model.layers[-4:]:
        layer.trainable = True


def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', keras.metrics.AUC(name='auc'),
                           keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')])

### Training

In [ ]:
class MacroF1(keras.callbacks.Callback):
    # macro F1 dan macro recall validasi tiap epoch, dipakai untuk early stopping
    def __init__(self, val_ds):
        super().__init__()
        self.val_ds = val_ds

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for x, y in self.val_ds:
            y_pred.append(np.argmax(self.model.predict_on_batch(x), axis=1))
            y_true.append(np.argmax(y, axis=1))
        y_true, y_pred = np.concatenate(y_true), np.concatenate(y_pred)
        logs['val_f1_macro'] = f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
        logs['val_recall_macro'] = recall_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)
        print(f" - val_f1_macro: {logs['val_f1_macro']:.4f} - val_recall_macro: {logs['val_recall_macro']:.4f}")


def get_callbacks(val_ds, patience, reduce_lr=False):
    cb = [MacroF1(val_ds),
          keras.callbacks.EarlyStopping(monitor='val_f1_macro', mode='max', patience=patience,
                                        restore_best_weights=True, verbose=1)]
    if reduce_lr:
        cb.append(keras.callbacks.ReduceLROnPlateau(monitor='val_f1_macro', mode='max', factor=0.5,
                                                    patience=5, min_lr=1e-8, verbose=1))
    return cb


def train_frozen(train_ds, val_ds, class_weight=None):
    # fase 1: backbone beku, hanya head yang dilatih
    model = build_model()
    set_trainable(model, False)
    compile_model(model, 1e-3)
    h = model.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=class_weight,
                  callbacks=get_callbacks(val_ds, patience=5))
    return model, h.history


def train_finetune(train_ds, val_ds, class_weight, lr_phase2):
    # fase 1 lalu fase 2: semua layer dibuka kecuali BatchNorm
    model, h1 = train_frozen(train_ds, val_ds, class_weight)
    set_trainable(model, True)
    compile_model(model, lr_phase2)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=50, class_weight=class_weight,
                   callbacks=get_callbacks(val_ds, patience=10, reduce_lr=True))
    return model, {'phase1': h1, 'phase2': h2.history}


def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2, default=float)


def load_json(path):
    with open(path) as f:
        return json.load(f)

In [ ]:
def specificity(cm):
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    tn = cm.sum() - tp - fp - fn
    return tn / (tn + fp)


def evaluate(y_true, y_prob):
    y_pred = y_prob.argmax(axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0),
        'specificity_macro': specificity(cm).mean(),
        'auc_macro': roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro', labels=LABELS),
        'missed_mel': int(((y_true == MEL) & (y_pred != MEL)).sum()),
    }


def per_class(y_true, y_prob):
    p, r, f, s = precision_recall_fscore_support(y_true, y_prob.argmax(axis=1), labels=LABELS, zero_division=0)
    return pd.DataFrame({'class': CLASS_NAMES, 'n': s, 'precision': p, 'recall': r, 'f1': f}).round(4)


def fold_data(fold):
    tr, va = folds[fold - 1]
    return train_df.iloc[tr].reset_index(drop=True), train_df.iloc[va].reset_index(drop=True)


def monitor_split(data):
    # 10% data train (per lesi) hanya untuk early stopping model final
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
    fit_idx, mon_idx = next(sgkf.split(data, data['label'], groups=data['lesion_id']))
    return data.iloc[fit_idx].reset_index(drop=True), data.iloc[mon_idx].reset_index(drop=True)

### Model A: EfficientNetV2S tanpa fine-tuning (backbone beku)

5-fold CV. Fold yang sudah pernah dilatih (ada file prediksinya) langsung dimuat.

In [ ]:
probs_A = []
for fold in range(1, N_FOLDS + 1):
    fold_train, fold_val = fold_data(fold)
    meta_file = A_DIR / 'cv' / f'fold{fold}_metrics.json'
    prob_file = A_DIR / 'cv' / f'fold{fold}_y_prob.npy'

    if meta_file.exists() and prob_file.exists():
        assert sorted(load_json(meta_file)['val_image_ids']) == sorted(fold_val['image_id'].astype(str))
        print(f'Fold {fold}: load dari checkpoint')
        probs_A.append(np.load(prob_file))
        continue

    print(f'\nFold {fold}')
    keras.backend.clear_session()
    set_seed(SEED + fold)
    train_ds = make_dataset(fold_train, shuffle=True)
    val_ds = make_dataset(fold_val)
    model, hist = train_frozen(train_ds, val_ds)

    prob = model.predict(val_ds, verbose=0)
    np.save(prob_file, prob)
    save_json({'phase1': hist}, A_DIR / 'cv' / f'fold{fold}_history.json')
    save_json({'fold': fold, 'val_image_ids': list(fold_val['image_id'].astype(str))}, meta_file)
    probs_A.append(prob)

    del model
    gc.collect()

In [ ]:
# model final A: 90% data train untuk training, 10% untuk early stopping
final_A = A_DIR / 'final'

if (final_A / 'y_prob.npy').exists():
    prob_test_A = np.load(final_A / 'y_prob.npy')
    print('Model final A sudah ada, prediksi test dimuat')
else:
    fit_df, mon_df = monitor_split(train_df)
    keras.backend.clear_session()
    set_seed(SEED)
    model, hist = train_frozen(make_dataset(fit_df, shuffle=True), make_dataset(mon_df))
    model.save(final_A / 'model_A_final.keras')
    save_json({'phase1': hist}, final_A / 'history.json')

    prob_test_A = model.predict(make_dataset(test_df), verbose=0)
    np.save(final_A / 'y_true.npy', test_df['label'].values)
    np.save(final_A / 'y_prob.npy', prob_test_A)

### Model B: EfficientNetV2S fine-tuned (augmentasi + class weight)

Learning rate fase 2 dipilih dari {0.0001, 0.0003} dengan 5-fold CV (kriteria: macro recall).

In [ ]:
TAG = 'S4_EfficientNetV2S'
LR_GRID = [1e-4, 3e-4]


def hp_dir(lr):
    hp = {'dense_units': 256, 'dropout_rate': 0.5, 'phase1_lr': 1e-3, 'phase2_lr': lr}
    key = hashlib.md5(json.dumps(hp, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return CHECKPOINT_DIR / TAG / f'hp_{key}'


probs_B = {}
for lr in LR_GRID:
    d = hp_dir(lr)
    d.mkdir(parents=True, exist_ok=True)
    probs_B[lr] = []
    for fold in range(1, N_FOLDS + 1):
        fold_train, fold_val = fold_data(fold)
        val_ids = list(fold_val['image_id'].astype(str))
        meta_file = d / f'fold{fold}_metrics.json'
        prob_file = d / f'fold{fold}_y_prob.npy'

        if meta_file.exists() and prob_file.exists():
            saved_ids = [str(i) for i in load_json(meta_file)['val_image_ids']]
            assert sorted(saved_ids) == sorted(val_ids)
            order = {img: i for i, img in enumerate(saved_ids)}
            probs_B[lr].append(np.load(prob_file)[[order[i] for i in val_ids]])
            print(f'lr {lr} fold {fold}: load dari checkpoint')
            continue

        print(f'\nlr {lr} | Fold {fold}')
        keras.backend.clear_session()
        set_seed(SEED + fold)
        train_ds = make_dataset(fold_train, augment=True, shuffle=True)
        val_ds = make_dataset(fold_val)
        model, hist = train_finetune(train_ds, val_ds, get_class_weight(fold_train), lr)

        prob = model.predict(val_ds, verbose=0)
        np.save(prob_file, prob)
        save_json(hist, d / f'fold{fold}_history.json')
        save_json({'fold': fold, 'val_image_ids': val_ids}, meta_file)
        probs_B[lr].append(prob)

        del model
        gc.collect()

In [ ]:
grid = []
for lr in LR_GRID:
    res = pd.DataFrame([evaluate(fold_data(f)[1]['label'].values, p)
                        for f, p in enumerate(probs_B[lr], 1)])
    grid.append({'lr_phase2': lr, **res.mean().round(4).to_dict()})
grid = pd.DataFrame(grid)
best_lr = grid.loc[grid['recall_macro'].idxmax(), 'lr_phase2']
print('learning rate terpilih:', best_lr)
grid

In [ ]:
# model final B dengan learning rate terpilih
final_B_pred = METRICS_DIR / f'{TAG}_y_prob.npy'

if final_B_pred.exists():
    prob_test_B = np.load(final_B_pred)
    print('Model final B sudah ada, prediksi test dimuat')
else:
    fit_df, mon_df = monitor_split(train_df)
    keras.backend.clear_session()
    set_seed(SEED)
    model, hist = train_finetune(make_dataset(fit_df, augment=True, shuffle=True), make_dataset(mon_df),
                                 get_class_weight(fit_df), best_lr)
    model.save(MODEL_DIR / f'{TAG}_final.keras')
    save_json(hist, HISTORY_DIR / f'{TAG}_final_history.json')

    prob_test_B = model.predict(make_dataset(test_df), verbose=0)
    np.save(METRICS_DIR / f'{TAG}_y_true.npy', test_df['label'].values)
    np.save(final_B_pred, prob_test_B)

### Hasil 5-Fold Cross Validation

In [ ]:
rows = []
for name, probs in [('A', probs_A), ('B', probs_B[best_lr])]:
    for fold, p in enumerate(probs, 1):
        rows.append({'model': name, 'fold': fold, **evaluate(fold_data(fold)[1]['label'].values, p)})

cv_results = pd.DataFrame(rows).round(4)
cv_results.to_csv(OUT_DIR / 'cv_per_fold.csv', index=False)
cv_results

In [ ]:
cv_results.drop(columns=['fold', 'missed_mel']).groupby('model').agg(['mean', 'std', 'min', 'max']).round(4).T

In [ ]:
print('Melanoma terlewat (total 5 fold):', cv_results.groupby('model')['missed_mel'].sum().to_dict())
a = cv_results[cv_results['model'] == 'A'].reset_index(drop=True)
b = cv_results[cv_results['model'] == 'B'].reset_index(drop=True)
print('Jumlah fold B > A:')
print((b.drop(columns=['model', 'fold', 'missed_mel']) > a.drop(columns=['model', 'fold', 'missed_mel'])).sum())

### Hasil Test Set

In [ ]:
y_test = test_df['label'].values

test_results = pd.DataFrame({'Model A': evaluate(y_test, prob_test_A),
                             'Model B': evaluate(y_test, prob_test_B)})
test_results['B - A'] = test_results['Model B'] - test_results['Model A']
test_results.round(4)

In [ ]:
print('Model A')
print(classification_report(y_test, prob_test_A.argmax(1), target_names=CLASS_NAMES, digits=4))
print('Model B')
print(classification_report(y_test, prob_test_B.argmax(1), target_names=CLASS_NAMES, digits=4))

In [ ]:
per_class(y_test, prob_test_A).merge(per_class(y_test, prob_test_B), on=['class', 'n'], suffixes=('_A', '_B'))

In [ ]:
cm_A = confusion_matrix(y_test, prob_test_A.argmax(1), labels=LABELS)
cm_B = confusion_matrix(y_test, prob_test_B.argmax(1), labels=LABELS)
print('Confusion matrix A')
print(pd.DataFrame(cm_A, index=CLASS_NAMES, columns=CLASS_NAMES))
print('\nConfusion matrix B')
print(pd.DataFrame(cm_B, index=CLASS_NAMES, columns=CLASS_NAMES))

### Uji Statistik (B - A)

Bootstrap berpasangan 2000 kali. Unit resampling bisa per citra atau per lesi (citra dari lesi yang sama ikut terambil bersama). Accuracy juga diuji dengan McNemar (biasa dan versi berklaster per lesi, Durkalski dkk. 2003).

In [ ]:
def macro_scores(y, pred):
    cm = np.bincount(y * NUM_CLASSES + pred, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    tp = np.diag(cm).astype(float)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    pre = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    f1 = np.divide(2 * pre * rec, pre + rec, out=np.zeros_like(tp), where=(pre + rec) > 0)
    return np.array([rec.mean(), f1.mean(), (y == pred).mean()])


def bootstrap_test(y, prob_a, prob_b, groups, n_boot=2000):
    # groups = unit resampling (indeks citra atau lesion_id), diambil ulang di dalam tiap kelas
    pa, pb = prob_a.argmax(1), prob_b.argmax(1)
    g = pd.DataFrame({'unit': groups, 'y': y})
    members = g.groupby('unit').indices
    unit_label = g.groupby('unit')['y'].first()
    units_per_class = [unit_label.index[unit_label.values == c].values for c in LABELS]

    rng = np.random.default_rng(SEED)
    diffs = np.empty((n_boot, 3))
    for i in range(n_boot):
        picked = np.concatenate([rng.choice(u, len(u), replace=True) for u in units_per_class])
        idx = np.concatenate([members[u] for u in picked])
        diffs[i] = macro_scores(y[idx], pb[idx]) - macro_scores(y[idx], pa[idx])

    point = macro_scores(y, pb) - macro_scores(y, pa)
    out = []
    for j, name in enumerate(['macro recall', 'macro F1', 'accuracy']):
        v = diffs[:, j]
        p = min(1.0, max(1 / n_boot, 2 * min((v <= 0).mean(), (v >= 0).mean())))
        out.append({'metric': name, 'diff': point[j], 'ci_low': np.percentile(v, 2.5),
                    'ci_high': np.percentile(v, 97.5), 'p': p})
    return pd.DataFrame(out).round(4)


def mcnemar_test(y, prob_a, prob_b, groups=None):
    a_ok, b_ok = prob_a.argmax(1) == y, prob_b.argmax(1) == y
    b, c = int((b_ok & ~a_ok).sum()), int((a_ok & ~b_ok).sum())
    if groups is None:
        return {'B only': b, 'A only': c, 'p': float(binomtest(min(b, c), b + c, 0.5).pvalue)}
    # versi berklaster (Durkalski dkk., 2003)
    d = pd.DataFrame({'g': groups, 'diff': (b_ok & ~a_ok).astype(int) - (a_ok & ~b_ok).astype(int)})
    dk = d.groupby('g')['diff'].sum().values
    x2 = dk.sum() ** 2 / (dk ** 2).sum()
    return {'B only': b, 'A only': c, 'chi2': round(float(x2), 2), 'p': float(chi2.sf(x2, 1))}

In [ ]:
lesion_test = test_df['lesion_id'].astype(str).values
print('Test set:', len(y_test), 'citra,', len(set(lesion_test)), 'lesi')

print('\nBootstrap per lesi')
display(bootstrap_test(y_test, prob_test_A, prob_test_B, lesion_test))
print('McNemar berklaster:', mcnemar_test(y_test, prob_test_A, prob_test_B, lesion_test))

print('\nBootstrap per citra')
display(bootstrap_test(y_test, prob_test_A, prob_test_B, np.arange(len(y_test))))
print('McNemar:', mcnemar_test(y_test, prob_test_A, prob_test_B))

In [ ]:
# prediksi out-of-fold CV (eksploratif)
y_oof = np.concatenate([fold_data(f)[1]['label'].values for f in range(1, N_FOLDS + 1)])
lesion_oof = np.concatenate([fold_data(f)[1]['lesion_id'].astype(str).values for f in range(1, N_FOLDS + 1)])
oof_A = np.concatenate(probs_A)
oof_B = np.concatenate(probs_B[best_lr])
print('CV:', len(y_oof), 'citra,', len(set(lesion_oof)), 'lesi')

display(bootstrap_test(y_oof, oof_A, oof_B, lesion_oof))
print('McNemar berklaster:', mcnemar_test(y_oof, oof_A, oof_B, lesion_oof))

In [ ]:
# cek run lr fase 2 = 0.0003: distribusi kelas hasil prediksi per fold
rows = []
for fold, p in enumerate(probs_B[3e-4], 1):
    y = fold_data(fold)[1]['label'].values
    pred = p.argmax(1)
    rows.append({'fold': fold, **dict(zip(CLASS_NAMES, np.bincount(pred, minlength=NUM_CLASSES))),
                 'nan': int((~np.isfinite(p)).any(axis=1).sum()),
                 'accuracy': (pred == y).mean(),
                 'recall_macro': recall_score(y, pred, labels=LABELS, average='macro', zero_division=0),
                 'f1_macro': f1_score(y, pred, labels=LABELS, average='macro', zero_division=0)})
pd.DataFrame(rows).round(4)

### Gambar

In [ ]:
COLOR_A, COLOR_B = '#2a78d6', '#eb6834'
plt.rcParams.update({'font.family': 'serif', 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})

# distribusi kelas train/test
fig, ax = plt.subplots(figsize=(3.5, 2.4))
x = np.arange(NUM_CLASSES)
n_train = train_df['dx'].value_counts().reindex(CLASS_NAMES).values
n_test = test_df['dx'].value_counts().reindex(CLASS_NAMES).values
ax.bar(x - .2, n_train, .38, color=COLOR_A, label='Train')
ax.bar(x + .2, n_test, .38, color=COLOR_B, label='Test')
for i in x:
    ax.text(i - .2, n_train[i] * 1.15, n_train[i], ha='center', va='bottom', fontsize=5.5, rotation=90)
    ax.text(i + .2, n_test[i] * 1.15, n_test[i], ha='center', va='bottom', fontsize=5.5, rotation=90)
ax.set_yscale('log')
ax.set_ylim(10, 4e4)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Number of images (log)')
ax.legend(frameon=False, fontsize=7, ncol=2, loc='upper left')
fig.savefig(FIG_DIR / 'fig1_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# macro F1 dan macro recall per fold
fig, axes = plt.subplots(1, 2, figsize=(3.5, 2.7))
for ax, metric, title in zip(axes, ['f1_macro', 'recall_macro'], ['(a) Macro F1-score', '(b) Macro recall']):
    for j, (name, color, marker) in enumerate([('A', COLOR_A, 'o'), ('B', COLOR_B, '^')]):
        v = cv_results.loc[cv_results['model'] == name, metric].values
        ax.scatter(j + np.linspace(-.16, .16, len(v)), v, s=34, marker=marker, color=color,
                   edgecolor='white', linewidth=.6, zorder=3)
        ax.hlines(v.mean(), j - .3, j + .3, color='black', lw=1.8, zorder=4)
        ax.text(j + .33, v.mean(), f'{v.mean():.3f}', va='center', fontsize=7)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Model A', 'Model B'], fontsize=8)
    ax.set_xlim(-.55, 1.75)
    ax.set_ylim(.40, .85)
    ax.set_title(title, fontsize=8.5, loc='left')
    ax.grid(axis='y', alpha=.3)
axes[0].set_ylabel('Value (per fold)')
axes[1].set_yticklabels([])
handles = [plt.Line2D([], [], marker='o', ls='', color=COLOR_A, label='Model A fold'),
           plt.Line2D([], [], marker='^', ls='', color=COLOR_B, label='Model B fold'),
           plt.Line2D([], [], color='black', lw=1.8, label='Mean of 5 folds')]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=3, frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig2_cv_per_fold.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# recall dan F1 per kelas di test set
pc_A, pc_B = per_class(y_test, prob_test_A), per_class(y_test, prob_test_B)
fig, axes = plt.subplots(2, 1, figsize=(3.5, 3.6), sharex=True)
for ax, metric, title in zip(axes, ['recall', 'f1'], ['(a) Recall', '(b) F1-score']):
    ax.bar(x - .2, pc_A[metric], .38, color=COLOR_A, label='Model A')
    ax.bar(x + .2, pc_B[metric], .38, color=COLOR_B, label='Model B')
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=8, loc='left')
    ax.grid(axis='y', alpha=.3)
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_NAMES)
axes[0].legend(frameon=False, fontsize=6.5, ncol=2, loc='lower center', bbox_to_anchor=(.5, 1.2))
fig.savefig(FIG_DIR / 'fig3_per_class.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# confusion matrix test set (jumlah dan persen per baris)
fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.45))
for ax, title, cm in zip(axes, ['(a) Model A', '(b) Model B'], [cm_A, cm_B]):
    row_pct = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(row_pct, cmap='Blues', vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            color = 'white' if row_pct[i, j] > .55 else 'black'
            ax.text(j, i - .16, cm[i, j], ha='center', va='center', fontsize=8, fontweight='bold', color=color)
            ax.text(j, i + .22, f'{row_pct[i, j] * 100:.0f}%', ha='center', va='center', fontsize=6.5, color=color)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title(title, fontsize=9, loc='left')
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xticks(np.arange(-.5, NUM_CLASSES, 1), minor=True)
    ax.set_yticks(np.arange(-.5, NUM_CLASSES, 1), minor=True)
    ax.grid(which='minor', color='white', lw=1)
    ax.tick_params(which='minor', length=0)
fig.tight_layout(rect=(0, 0, .92, 1))
cax = fig.add_axes([.935, .2, .015, .65])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label('Share of true-class row', fontsize=7.5)
cbar.ax.tick_params(labelsize=7)
fig.savefig(FIG_DIR / 'fig4_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()